# v09 — Backtest: Kill-switch drawdown harian/bulanan/total

**Latar belakang:** kode live (`app/features/m5_scalping/usecase.py`, `_check_drawdown_guard`)
sudah punya kill-switch drawdown dengan nilai default `.env.example`: **daily 5%, monthly 10%,
total 15%**. Nilai ini belum divalidasi — sebelum dipakai live, kita cek dulu apakah angka ini
masuk akal berdasarkan data historis v06, atau apakah kombinasi lain lebih baik.

**Cara kerja kill-switch di kode live** (disimulasikan persis sama di sini):
- **Daily**: dibandingkan ke equity awal hari (UTC) — reset otomatis begitu tanggal berganti
- **Monthly**: dibandingkan ke equity awal bulan (UTC) — reset otomatis begitu bulan berganti
- **Total**: dibandingkan ke peak equity tertinggi yang pernah tercatat — begitu breach, **pause
  PERMANEN** (tidak reset otomatis, disimulasikan sbg berhenti total sampai akhir data)
- Begitu salah satu ke-trigger, robot **skip entry baru** sampai reset (peak/baseline TIDAK
  berubah oleh trade yang di-skip, karena tidak ada trade yang terjadi)

**Metodologi:** replay 1664 trade v06 (`dataset/processed/m5_scalping/v06/trade_log_full.csv`)
secara berurutan, tapi kali ini terapkan kill-switch — kalau lagi "paused", trade itu **di-skip**
(tidak dieksekusi, equity tidak berubah karena trade itu). Bandingkan equity curve & metrik akhir
dengan skenario TANPA kill-switch (v06 asli), untuk beberapa kombinasi threshold.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.append(str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

STRATEGY_NAME = "m5_scalping"
VERSION = "v09"

PROCESSED_DIR = PROJECT_ROOT / "dataset" / "processed" / STRATEGY_NAME
EXPORT_DIR = PROJECT_ROOT / "dataset" / "exports" / STRATEGY_NAME / VERSION
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.width", 160)
plt.rcParams["figure.figsize"] = (14, 5)

## 1. Load trade log v06 (baseline, TANPA kill-switch)

In [2]:
trades = pd.read_csv(PROCESSED_DIR / "v06" / "trade_log_full.csv")
trades["entry_time"] = pd.to_datetime(trades["entry_time"])
trades["exit_time"] = pd.to_datetime(trades["exit_time"])
trades = trades.sort_values("entry_time").reset_index(drop=True)

INITIAL_EQUITY = 100.0
print(f"Total trade v06 (baseline): {len(trades)}")
print(f"Equity awal: ${INITIAL_EQUITY}, akhir (tanpa kill-switch): ${trades['equity_after'].iloc[-1]:.2f}")

baseline_equity_series = pd.Series([INITIAL_EQUITY] + trades["pnl"].cumsum().add(INITIAL_EQUITY).tolist())
baseline_running_max = baseline_equity_series.cummax()
baseline_dd = (baseline_equity_series - baseline_running_max) / baseline_running_max * 100
print(f"Max drawdown historis TANPA kill-switch: {baseline_dd.min():.2f}%")

Total trade v06 (baseline): 1664
Equity awal: $100.0, akhir (tanpa kill-switch): $2425.04
Max drawdown historis TANPA kill-switch: -28.43%


## 2. Simulator kill-switch — replay trade v06 dengan guard daily/monthly/total

Logic ini adalah port 1:1 dari `_check_drawdown_guard()` di `app/features/m5_scalping/usecase.py`,
supaya hasil simulasi representatif terhadap perilaku kode live yang sebenarnya.

In [3]:
def simulate_with_killswitch(
    trades_df: pd.DataFrame,
    initial_equity: float,
    max_daily_pct: float,
    max_monthly_pct: float,
    max_total_pct: float,
) -> pd.DataFrame:
    """Replay trade berurutan; kalau kill-switch aktif SEBELUM entry_time trade tsb (dicek
    pakai equity & baseline SAAT itu, sebelum trade dieksekusi), trade di-skip (executed=False,
    equity tidak berubah). Baseline daily/monthly direset begitu tanggal/bulan kalender (UTC)
    berbeda dari trade sebelumnya -- sama seperti kode live yang cek tiap polling.
    """
    equity = initial_equity
    peak_equity = initial_equity
    day_key = None
    day_start_equity = initial_equity
    month_key = None
    month_start_equity = initial_equity
    total_dd_paused = False

    records = []
    for _, tr in trades_df.iterrows():
        now = tr["entry_time"]
        dk = now.strftime("%Y-%m-%d")
        mk = now.strftime("%Y-%m")

        if day_key is None:
            day_key, day_start_equity = dk, equity
        elif day_key != dk:
            day_key, day_start_equity = dk, equity

        if month_key is None:
            month_key, month_start_equity = mk, equity
        elif month_key != mk:
            month_key, month_start_equity = mk, equity

        total_dd_pct = (peak_equity - equity) / peak_equity * 100 if peak_equity > 0 else 0.0
        daily_dd_pct = (day_start_equity - equity) / day_start_equity * 100 if day_start_equity > 0 else 0.0
        monthly_dd_pct = (month_start_equity - equity) / month_start_equity * 100 if month_start_equity > 0 else 0.0

        paused_reason = None
        if total_dd_paused or total_dd_pct >= max_total_pct:
            total_dd_paused = True
            paused_reason = "TOTAL"
        elif daily_dd_pct >= max_daily_pct:
            paused_reason = "DAILY"
        elif monthly_dd_pct >= max_monthly_pct:
            paused_reason = "MONTHLY"

        executed = paused_reason is None
        if executed:
            equity += tr["pnl"]
            if equity > peak_equity:
                peak_equity = equity

        records.append({
            "entry_time": now, "pnl": tr["pnl"] if executed else 0.0,
            "original_pnl": tr["pnl"], "executed": executed, "paused_reason": paused_reason,
            "equity": equity, "peak_equity": peak_equity,
            "daily_dd_pct": daily_dd_pct, "monthly_dd_pct": monthly_dd_pct, "total_dd_pct": total_dd_pct,
        })

    return pd.DataFrame(records)


def evaluate_killswitch_run(sim: pd.DataFrame, initial_equity: float) -> dict:
    executed = sim[sim["executed"]]
    skipped = sim[~sim["executed"]]
    equity_series = pd.Series([initial_equity] + sim["equity"].tolist())
    running_max = equity_series.cummax()
    dd = (equity_series - running_max) / running_max * 100

    wins = executed[executed["pnl"] > 0]
    gross_profit = wins["pnl"].sum()
    gross_loss = executed[executed["pnl"] <= 0]["pnl"].sum()

    return {
        "total_trades": len(sim),
        "executed_trades": len(executed),
        "skipped_trades": len(skipped),
        "skipped_pct": round(len(skipped) / len(sim) * 100, 1),
        "win_rate_pct": round((executed["pnl"] > 0).mean() * 100, 2) if len(executed) else 0,
        "profit_factor": round(gross_profit / abs(gross_loss), 2) if gross_loss != 0 else float("inf"),
        "final_equity": round(sim["equity"].iloc[-1], 2) if len(sim) else initial_equity,
        "net_pnl": round(sim["equity"].iloc[-1] - initial_equity, 2) if len(sim) else 0,
        "max_drawdown_pct": round(dd.min(), 2),
        "min_equity": round(equity_series.min(), 2),
        "n_daily_pauses": (skipped["paused_reason"] == "DAILY").sum(),
        "n_monthly_pauses": (skipped["paused_reason"] == "MONTHLY").sum(),
        "n_total_pauses": (skipped["paused_reason"] == "TOTAL").sum(),
        "total_dd_ever_triggered": bool((skipped["paused_reason"] == "TOTAL").any()),
    }

## 3. Grid search kombinasi threshold (daily x monthly x total)

Termasuk kandidat default `.env.example` (5/10/15) sbg salah satu titik, dibandingkan dgn
kombinasi lain -- lebih ketat & lebih longgar -- utk lihat trade-off nyata.

In [4]:
daily_candidates = [3.0, 5.0, 7.0, 10.0]
monthly_candidates = [8.0, 10.0, 15.0, 20.0]
total_candidates = [12.0, 15.0, 20.0, 30.0]

grid_results = []
for daily in daily_candidates:
    for monthly in monthly_candidates:
        if monthly <= daily:
            continue  # monthly harus lebih longgar dari daily (gak masuk akal sebaliknya)
        for total in total_candidates:
            if total <= monthly:
                continue  # total harus lebih longgar dari monthly
            sim = simulate_with_killswitch(trades, INITIAL_EQUITY, daily, monthly, total)
            metrics = evaluate_killswitch_run(sim, INITIAL_EQUITY)
            metrics.update({"daily_pct": daily, "monthly_pct": monthly, "total_pct": total})
            grid_results.append(metrics)

grid_df = pd.DataFrame(grid_results)
print(f"Total kombinasi diuji: {len(grid_df)}")
grid_df.sort_values("final_equity", ascending=False).head(15)

Total kombinasi diuji: 36


,total_trades,executed_trades,skipped_trades,skipped_pct,win_rate_pct,profit_factor,final_equity,net_pnl,max_drawdown_pct,min_equity,n_daily_pauses,n_monthly_pauses,n_total_pauses,total_dd_ever_triggered,daily_pct,monthly_pct,total_pct
33,1664,157,1507,90.6,50.96,1.12,126.57,26.57,-20.37,88.15,0,0,1507,True,10.0,15.0,20.0
19,1664,150,1514,91.0,50.67,1.08,116.97,16.97,-21.67,88.15,7,0,1507,True,5.0,15.0,20.0
30,1664,153,1511,90.8,50.33,1.08,116.56,16.56,-21.73,88.15,4,0,1507,True,7.0,15.0,20.0
8,1664,133,1531,92.0,50.38,1.08,115.02,15.02,-21.49,88.15,22,0,1509,True,3.0,15.0,20.0
5,1664,49,1615,97.1,44.90,1.20,114.98,14.98,-15.83,88.15,15,81,1519,True,3.0,10.0,15.0
1,1664,49,1615,97.1,44.90,1.20,114.98,14.98,-15.83,88.15,15,81,1519,True,3.0,8.0,15.0
35,1664,269,1395,83.8,48.70,1.02,111.65,11.65,-31.14,88.15,6,55,1334,True,10.0,20.0,30.0
34,1664,180,1484,89.2,49.44,1.03,108.76,8.76,-31.57,88.15,0,59,1425,True,10.0,15.0,30.0
17,1664,69,1595,95.9,46.38,1.05,106.00,6.00,-21.68,88.15,5,81,1509,True,5.0,10.0,20.0
13,1664,69,1595,95.9,46.38,1.05,106.00,6.00,-21.68,88.15,5,81,1509,True,5.0,8.0,20.0


## 4. Bandingkan: default `.env.example` (5/10/15) vs TANPA kill-switch vs kandidat terbaik

In [5]:
sim_default = simulate_with_killswitch(trades, INITIAL_EQUITY, 5.0, 10.0, 15.0)
metrics_default = evaluate_killswitch_run(sim_default, INITIAL_EQUITY)

metrics_no_killswitch = {
    "total_trades": len(trades), "executed_trades": len(trades), "skipped_trades": 0, "skipped_pct": 0.0,
    "win_rate_pct": round((trades["pnl"] > 0).mean() * 100, 2),
    "profit_factor": round(trades[trades["pnl"] > 0]["pnl"].sum() / abs(trades[trades["pnl"] <= 0]["pnl"].sum()), 2),
    "final_equity": round(baseline_equity_series.iloc[-1], 2),
    "net_pnl": round(baseline_equity_series.iloc[-1] - INITIAL_EQUITY, 2),
    "max_drawdown_pct": round(baseline_dd.min(), 2),
    "min_equity": round(baseline_equity_series.min(), 2),
    "n_daily_pauses": 0, "n_monthly_pauses": 0, "n_total_pauses": 0, "total_dd_ever_triggered": False,
}

best_row = grid_df.sort_values("final_equity", ascending=False).iloc[0]
sim_best = simulate_with_killswitch(
    trades, INITIAL_EQUITY, best_row["daily_pct"], best_row["monthly_pct"], best_row["total_pct"]
)
metrics_best = evaluate_killswitch_run(sim_best, INITIAL_EQUITY)

comparison = pd.DataFrame({
    "TANPA kill-switch (v06 asli)": metrics_no_killswitch,
    "Default .env (5/10/15)": metrics_default,
    f"Terbaik grid ({best_row['daily_pct']}/{best_row['monthly_pct']}/{best_row['total_pct']})": metrics_best,
})
comparison

,TANPA kill-switch (v06 asli),Default .env (5/10/15),Terbaik grid (10.0/15.0/20.0)
total_trades,1664,1664,1664
executed_trades,1664,16,157
skipped_trades,0,1648,1507
skipped_pct,0.0,99.0,90.6
win_rate_pct,50.18,37.5,50.96
profit_factor,1.35,0.74,1.12
final_equity,2425.04,92.32,126.57
net_pnl,2325.04,-7.68,26.57
max_drawdown_pct,-28.43,-15.37,-20.37
min_equity,88.15,88.15,88.15


## 5. Interpretasi trade-off

Kill-switch itu **bukan** alat buat menaikkan profit — tujuannya membatasi kerugian ekstrem
(tail risk), jadi wajar kalau `final_equity`/`net_pnl` sedikit lebih rendah dibanding tanpa
kill-switch (karena beberapa trade WIN yang kebetulan terjadi saat drawdown lagi "dalam" juga
ikut ke-skip). Yang harus dilihat adalah **`max_drawdown_pct`** (turun signifikan dari baseline)
dan **`min_equity`** (seberapa dekat modal ke nol) — itu ukuran keberhasilan sebenarnya, bukan
net_pnl.

In [ ]:
fig, ax = plt.subplots()
ax.plot(baseline_equity_series.to_numpy(dtype=np.float64), label="Tanpa kill-switch", alpha=0.7, linewidth=1.0)
ax.plot(
    [INITIAL_EQUITY] + sim_default["equity"].tolist(),
    label="Default .env (5/10/15)", alpha=0.9, linewidth=1.2,
)
ax.axhline(INITIAL_EQUITY, color="gray", linestyle="--", linewidth=0.8, label="Modal awal")
ax.set_title("Equity Curve: dengan vs tanpa kill-switch drawdown")
ax.set_xlabel("Trade ke-N")
ax.set_ylabel("Equity ($)")
ax.legend()
plt.tight_layout()
plt.savefig(EXPORT_DIR / "equity_curve_killswitch_comparison.png", dpi=120)
plt.show()

## 6. Detail kejadian pause (default 5/10/15) — seberapa sering & kapan

In [7]:
pauses = sim_default[~sim_default["executed"]].copy()
print(f"Total sinyal yang di-skip krn kill-switch: {len(pauses)} dari {len(sim_default)} ({len(pauses)/len(sim_default)*100:.1f}%)")
print()
print("Breakdown alasan:")
print(pauses["paused_reason"].value_counts())
print()
if pauses["paused_reason"].eq("TOTAL").any():
    first_total = pauses[pauses["paused_reason"] == "TOTAL"].iloc[0]
    print(f"PERINGATAN: total drawdown 15% ke-trigger pertama kali di {first_total['entry_time']}")
    print(f"Setelah ini SEMUA sinyal berikutnya di-skip permanen (perlu reset manual di kode live)")
else:
    print("Total drawdown 15% TIDAK PERNAH ke-trigger sepanjang backtest -- kill-switch total aman dipakai.")

Total sinyal yang di-skip krn kill-switch: 1648 dari 1664 (99.0%)

Breakdown alasan:
paused_reason
TOTAL      1562
MONTHLY      81
DAILY         5
Name: count, dtype: int64

PERINGATAN: total drawdown 15% ke-trigger pertama kali di 2025-02-07 01:10:00+00:00
Setelah ini SEMUA sinyal berikutnya di-skip permanen (perlu reset manual di kode live)


## 7. Simpan hasil

In [8]:
grid_df.to_csv(EXPORT_DIR / "killswitch_grid_search.csv", index=False)
sim_default.to_csv(EXPORT_DIR / "simulation_default_5_10_15.csv", index=False)
comparison.to_csv(EXPORT_DIR / "comparison_summary.csv")

with open(EXPORT_DIR / "metrics.txt", "w") as f:
    f.write("Backtest v09 — Kill-switch drawdown harian/bulanan/total\n\n")
    f.write("=== TANPA kill-switch (v06 asli) ===\n")
    for k, v in metrics_no_killswitch.items():
        f.write(f"  {k}: {v}\n")
    f.write("\n=== Default .env.example (daily=5%, monthly=10%, total=15%) ===\n")
    for k, v in metrics_default.items():
        f.write(f"  {k}: {v}\n")
    f.write(f"\n=== Terbaik dari grid search (daily={best_row['daily_pct']}%, "
            f"monthly={best_row['monthly_pct']}%, total={best_row['total_pct']}%) ===\n")
    for k, v in metrics_best.items():
        f.write(f"  {k}: {v}\n")

print("Tersimpan ke:", EXPORT_DIR)
for f in sorted(EXPORT_DIR.glob("*")):
    print(" -", f.name)

Tersimpan ke: D:\Projects\robot-scalping\dataset\exports\m5_scalping\v09
 - comparison_summary.csv
 - equity_curve_killswitch_comparison.png
 - killswitch_grid_search.csv
 - metrics.txt
 - simulation_default_5_10_15.csv
